In [29]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

In [30]:
# 2x2 board: positions 0 1 / 2 3
# Tiles 1, 2, 3 and blank = 0

start = (0, 1, 3, 2)
goal = (1, 2, 3, 0)

names = ["up", "down", "left", "right"]

In [31]:
def move(state, m):
    b = state.index(0)
    r, c = divmod(b, 2)

    if m == 0 and r == 1:
        n = b - 2
    elif m == 1 and r == 0:
        n = b + 2
    elif m == 2 and c == 1:
        n = b - 1
    elif m == 3 and c == 0:
        n = b + 1
    else:
        return state

    s = list(state)
    s[b], s[n] = s[n], s[b]

    return tuple(s)

In [32]:
target = None

for x in range(16):
    if move(move(start, x & 3), x >> 2) == goal:
        target = x

print("Target sequence index:", target)

Target sequence index: 7


In [33]:
n = 4

qc = QuantumCircuit(n, n)

qc.h(range(n))  # Create superposition

In [34]:
def flip_phase_of(qc, x):
    # Flip the sign of state |x>
    for q in range(n):
        if not (x >> q) & 1:
            qc.x(q)

    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)

    for q in range(n):
        if not (x >> q) & 1:
            qc.x(q)

In [35]:
for _ in range(3):

    # Oracle: mark the solution
    flip_phase_of(qc, target)

    # Diffuser
    qc.h(range(n))
    flip_phase_of(qc, 0)
    qc.h(range(n))

In [36]:
qc.measure(range(n), range(n))

print(qc.draw())

     ┌───┐               ┌───┐┌───┐                    ┌───┐┌───┐          »
q_0: ┤ H ├────────────■──┤ H ├┤ X ├─────────────────■──┤ X ├┤ H ├──────────»
     ├───┤            │  ├───┤├───┤                 │  ├───┤├───┤          »
q_1: ┤ H ├────────────■──┤ H ├┤ X ├─────────────────■──┤ X ├┤ H ├──────────»
     ├───┤            │  ├───┤├───┤                 │  ├───┤├───┤          »
q_2: ┤ H ├────────────■──┤ H ├┤ X ├─────────────────■──┤ X ├┤ H ├──────────»
     ├───┤┌───┐┌───┐┌─┴─┐├───┤├───┤┌───┐┌───┐┌───┐┌─┴─┐├───┤├───┤┌───┐┌───┐»
q_3: ┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├»
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘»
c: 4/══════════════════════════════════════════════════════════════════════»
                                                                           »
«               ┌───┐┌───┐                    ┌───┐┌───┐                    »
«q_0: ───────■──┤ H ├┤ X ├─────────────────■──┤ X ├┤ H ├─────────────────■─